# 01 - Datensatz und einzelne Messpunkte

In diesem ersten Schritt bauen wir nur die Datengrundlage auf. Es gibt
noch keine Mittelwerte, Steigungen, Zeitkonstanten oder kombinierten
Merkmale. Jeder Versuchsdatensatz enthält genau einen Rohmesspunkt.

Quelle: [Zenodo DOI 10.5281/zenodo.6821340](https://doi.org/10.5281/zenodo.6821340)

Fest: Sensor A, nur Kanal 0 und Sensorachse immer 1. UGM 1-500 werden
gruppiert in Train/Validation/Test geteilt. UGM 501-906 bleiben als
zusätzlicher exklusiver Extrapolationstest erhalten.

## Software und einstellbare Größen

Dieses Notebook führt in den Datenzugriff ein. Die eigentliche Logik liegt in
dataset_pipeline.py, damit dieselben Regeln später nicht versehentlich anders
implementiert werden.

**Verwendete Python-Werkzeuge**

- pathlib verwaltet Pfade unabhängig vom Startordner des Notebooks.
- urllib lädt die öffentliche MAT-Datei bei Bedarf von Zenodo.
- hashlib vergleicht die MD5-Prüfsumme und erkennt beschädigte Downloads.
- h5py liest die MATLAB-v7.3/HDF5-Datei, ohne MATLAB zu benötigen.
- NumPy stellt die Messzyklen als Arrays der Form (Zyklen, Sensoren, Samples)
  bereit und schreibt die kompakten NPZ-Exporte.
- scikit-learn liefert GroupShuffleSplit. Dadurch bleibt jede UGM-ID vollständig
  in genau einem Split.

**Wichtige Eingaben**

- transform = "stored" nutzt die auf Zenodo gespeicherten logarithmischen
  Sensorwerte; transform = "log1p" wendet zusätzlich NumPy log1p an.
- RANDOM_STATE legt die reproduzierbare Gruppenauswahl fest.
- BASE_TEST_SIZE und VALIDATION_SIZE_OF_DEVELOPMENT steuern die Anteile.
- PHYSICAL_SENSOR und SUB_SENSOR_INDEX bestimmen den Kanal. Für dieses Seminar
  bleiben sie absichtlich auf sensorA und 0; die Sensorachse muss Länge 1 haben.
- BASE_SEGMENT_MAX_UGM = 500 und EXTRA_TEST_MIN_UGM = 501 trennen den normalen
  Entwicklungsbereich vom exklusiven letzten Abschnitt.

Nach einer Änderung an Split-Konstanten müssen alle nachfolgenden Notebooks neu
ausgeführt werden. Test und test_extra dürfen niemals für Punktwahl,
Korrelation oder Hyperparameteroptimierung benutzt werden.

In [1]:
from pathlib import Path
import sys
import numpy as np

HERE = Path.cwd()
if not (HERE / "dataset_pipeline.py").exists():
    HERE = HERE / "Evaluation Seminar" / "Neuaufbau"
sys.path.insert(0, str(HERE.resolve()))
from dataset_pipeline import *

## 1. Quelle und Konzentrationsgrenze

Zenodo beschreibt für Aceton 3-50 ppb (UGM 1-200), 3-150 ppb
(UGM 201-500) und 3-500 ppb (UGM 501-906). Wir entfernen keine
Konzentrationen. Die ersten beiden Segmente bilden den üblichen
Interpolationstest; der letzte Abschnitt prüft exklusiv die Extrapolation
auf den erweiterten Konzentrationsraum.

In [2]:
path = ensure_dataset(download=True, verify=True)
print("Datensatz:", path)
print("Zenodo DOI:", ZENODO_DOI)

Datensatz: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\fullData.mat
Zenodo DOI: 10.5281/zenodo.6821340


## 2. Einen Kanal laden und den Split einfrieren

In [3]:
splits = prepare_splits("stored")
for name in SPLIT_NAMES:
    X = splits[name]["X"]
    targets = splits[name]["targets"]
    print(
        f"{name:>5}: X={X.shape}, "
        f"UGMs={len(np.unique(targets['range']))}, "
        f"Aceton={targets['acetone'].min():.1f}-"
        f"{targets['acetone'].max():.1f} ppb"
    )

train: X=(1090, 1, 1440), UGMs=318, Aceton=3.1-148.6 ppb
  val: X=(274, 1, 1440), UGMs=80, Aceton=4.5-150.0 ppb
 test: X=(338, 1, 1440), UGMs=100, Aceton=4.6-144.3 ppb
test_extra: X=(1392, 1, 1440), UGMs=402, Aceton=3.0-499.9 ppb


Die Form ist (Beispiele, 1, 1440). Kein Split enthält eine UGM-ID eines
anderen Splits. test_extra besteht ausschließlich aus UGM 501-906.

## 3. Nur Anfang und Ende einer Temperaturphase

In [4]:
points = temperature_boundary_points()
print("Erlaubte Punkte:", len(points))
for point in points[:8]:
    print(point)

Erlaubte Punkte: 48
BoundaryPoint(index=0, temperature_c=400, phase='high', cycle_step=0, edge='start')
BoundaryPoint(index=49, temperature_c=400, phase='high', cycle_step=0, edge='end')
BoundaryPoint(index=50, temperature_c=100, phase='low', cycle_step=0, edge='start')
BoundaryPoint(index=119, temperature_c=100, phase='low', cycle_step=0, edge='end')
BoundaryPoint(index=120, temperature_c=400, phase='high', cycle_step=1, edge='start')
BoundaryPoint(index=169, temperature_c=400, phase='high', cycle_step=1, edge='end')
BoundaryPoint(index=170, temperature_c=125, phase='low', cycle_step=1, edge='start')
BoundaryPoint(index=239, temperature_c=125, phase='low', cycle_step=1, edge='end')


Pro Zyklus gibt es zwölf 400-°C-Phasen mit je 5 s und zwölf niedrige
Phasen mit je 7 s. Die niedrigen Temperaturen steigen von 100 °C bis
375 °C. Aus jeder Phase sind nur Start und Ende zugelassen.

## 4. Korrelation und Querempfindlichkeit nur im Training

In [5]:
for gas in GAS_TARGETS:
    chosen = selected_point_scores(score_boundary_points(splits, gas))
    best = chosen["best_correlation"]
    selective = chosen["best_selectivity"]
    print(
        f"{gas:18s} | best: {best.point_id}, r={best.target_correlation:+.3f}"
        f" | selektiv: {selective.point_id}, "
        f"r={selective.target_correlation:+.3f}, "
        f"Störer={selective.strongest_interferer}, "
        f"Marge={selective.selectivity_margin:+.3f}"
    )

acetic_acid        | best: t325_low_step09_start_i1130, r=-0.416 | selektiv: t100_low_step00_start_i0050, r=-0.216, Störer=ethyl_acetate, Marge=-0.103
acetone            | best: t400_high_step02_end_i0289, r=-0.522 | selektiv: t150_low_step02_start_i0290, r=-0.338, Störer=ethyl_acetate, Marge=-0.059
carbon_monoxide    | best: t100_low_step00_end_i0119, r=-0.216 | selektiv: t100_low_step00_start_i0050, r=+0.026, Störer=ethyl_acetate, Marge=-0.292
ethanol            | best: t400_high_step02_start_i0240, r=-0.428 | selektiv: t100_low_step00_start_i0050, r=-0.209, Störer=ethyl_acetate, Marge=-0.110
ethyl_acetate      | best: t400_high_step03_end_i0409, r=-0.655 | selektiv: t400_high_step01_start_i0120, r=-0.622, Störer=acetone, Marge=+0.137
formaldehyde       | best: t275_low_step07_start_i0890, r=-0.351 | selektiv: t100_low_step00_start_i0050, r=-0.180, Störer=ethyl_acetate, Marge=-0.139
hydrogen           | best: t400_high_step11_start_i1320, r=-0.608 | selektiv: t400_high_step11_start_i

Die Selektivitätsmarge ist die Zielkorrelation minus stärkste Korrelation
zu einem anderen Gas oder zur Feuchte. Sie ist nur ein Screening-Wert,
kein Nachweis chemischer Selektivität.

## 5. Dasselbe Gas mit getrennten Punkten je Temperatur

In [6]:
target = "acetone"
selected = selected_point_scores(score_boundary_points(splits, target))
datasets = {
    name: build_point_dataset(splits, target, score)
    for name, score in selected.items()
}
for name, dataset in datasets.items():
    score = selected[name]
    print(
        f"{name:18s}: i={score.index:4d}, T={score.temperature_c:3d} °C, "
        f"{score.edge:5s}, r={score.target_correlation:+.3f}, "
        f"X_train={dataset['X_train'].shape}"
    )

best_correlation  : i= 289, T=400 °C, end  , r=-0.522, X_train=(1090, 1, 1)
best_selectivity  : i= 290, T=150 °C, start, r=-0.338, X_train=(1090, 1, 1)
temperature_100C  : i= 119, T=100 °C, end  , r=-0.403, X_train=(1090, 1, 1)
temperature_125C  : i= 239, T=125 °C, end  , r=-0.445, X_train=(1090, 1, 1)
temperature_150C  : i= 359, T=150 °C, end  , r=-0.446, X_train=(1090, 1, 1)
temperature_175C  : i= 479, T=175 °C, end  , r=-0.442, X_train=(1090, 1, 1)
temperature_200C  : i= 530, T=200 °C, start, r=-0.462, X_train=(1090, 1, 1)
temperature_225C  : i= 650, T=225 °C, start, r=-0.510, X_train=(1090, 1, 1)
temperature_250C  : i= 770, T=250 °C, start, r=-0.511, X_train=(1090, 1, 1)
temperature_275C  : i= 890, T=275 °C, start, r=-0.509, X_train=(1090, 1, 1)
temperature_300C  : i=1010, T=300 °C, start, r=-0.507, X_train=(1090, 1, 1)
temperature_325C  : i=1130, T=325 °C, start, r=-0.507, X_train=(1090, 1, 1)
temperature_350C  : i=1250, T=350 °C, start, r=-0.504, X_train=(1090, 1, 1)
temperature_

## 6. Alle Gas-/Punktvarianten exportieren

In [7]:
output_dir = project_root() / "Data" / "seminar_point_datasets"
manifest = export_point_datasets(output_dir)
print("Manifest:", manifest)
for transform in TRANSFORMS:
    raw_path, boundary_path = export_base_datasets(
        output_dir, transform
    )
    print(transform, "- alle Rohwerte:", raw_path)
    print(transform, "- alle Phasengrenzen:", boundary_path)

Manifest: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\seminar_point_datasets\manifest.csv
stored - alle Rohwerte: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\seminar_point_datasets\all_raw_values__stored__single_sensor.npz
stored - alle Phasengrenzen: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\seminar_point_datasets\all_temperature_boundaries__stored__single_sensor.npz
log1p - alle Rohwerte: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\seminar_point_datasets\all_raw_values__log1p__single_sensor.npz
log1p - alle Phasengrenzen: \\ROBNEXUS\Home\GIT\TCOCNN_PY\Data\seminar_point_datasets\all_temperature_boundaries__log1p__single_sensor.npz


Jeder Datensatz wird als stored und log1p erzeugt. stored bezeichnet die
Werte genau so, wie sie in Zenodo liegen; log1p ist eine zusätzliche
Transformation. Der Rohdatensatz hat (n, 1, 1440), der
Grenzpunktdatensatz (n, 1, 48).